# Univariate Time Series Workflow with chronobox

This notebook demonstrates a complete end-to-end pipeline for univariate time series analysis
using the **chronobox** library. We will work with Brazil's quarterly GDP data and cover:

1. Exploratory analysis and descriptive statistics
2. Unit root tests (ADF, KPSS, Zivot-Andrews)
3. STL decomposition
4. Economic filters (HP, Hamilton)
5. Model fitting (ARIMA, ETS, Auto-ARIMA, Holt-Winters)
6. Out-of-sample model comparison
7. Forecasting with the best model

**Dataset**: `brazil_gdp.csv` - Brazil quarterly real GDP (2000-2024)

## Pipeline Flowchart

```
Raw Data
   |
   v
[1. Exploratory Analysis] ---> Descriptive stats, visualization
   |
   v
[2. Stationarity Tests] ----> ADF, KPSS, Zivot-Andrews
   |                           |
   |                     Stationary? --> Yes --> Model in levels
   |                           |
   |                          No
   |                           |
   v                           v
[3. Decomposition] -------> STL (trend + seasonal + remainder)
   |
   v
[4. Filters] --------------> HP filter (trend/cycle)
   |                         Hamilton filter (trend/cycle)
   |
   v
[5. Train/Test Split] -----> Last 8 quarters as test set
   |
   v
[6. Model Fitting] --------> ARIMA(p,d,q)
   |                         ETS(error, trend, seasonal)
   |                         Auto-ARIMA (automatic selection)
   |                         Holt-Winters
   |
   v
[7. Model Comparison] -----> AIC, BIC, RMSE (out-of-sample)
   |
   v
[8. Forecast] -------------> Best model --> h-step-ahead forecast
   |                         with confidence intervals
   v
[9. Diagnostics] ----------> Residual analysis, Ljung-Box test
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# chronobox imports
from chronobox import ARIMA, ETS, HoltWinters, auto_arima
from chronobox.decomposition import STL
from chronobox.filters import hp_filter, hamilton_filter
from chronobox.tests_stat import adf_test, kpss_test, zivot_andrews_test
from chronobox.tests_stat import ljung_box_test, jarque_bera_test
from chronobox.visualization import (
    plot_series, plot_decomposition, plot_diagnostics, plot_forecast,
    set_theme
)

# Set professional theme
set_theme('professional')

print('chronobox imported successfully')

---
## 1. Data Loading and Exploratory Analysis

In [ ]:
# Load Brazil GDP data
import os
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
df = pd.read_csv(os.path.join(data_dir, 'brazil_gdp.csv'), parse_dates=['date'])
df = df.set_index('date')
df.index.freq = 'QS'

print(f'Period: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Observations: {len(df)}')
print(f'\nColumns: {list(df.columns)}')
df.head(10)

In [ ]:
# Descriptive statistics
y = df['gdp_real'].values

print('=== Descriptive Statistics ===')
print(f'Mean:     {np.mean(y):.2f}')
print(f'Std Dev:  {np.std(y):.2f}')
print(f'Min:      {np.min(y):.2f}')
print(f'Max:      {np.max(y):.2f}')
print(f'Skewness: {pd.Series(y).skew():.4f}')
print(f'Kurtosis: {pd.Series(y).kurtosis():.4f}')

# Growth rates
growth = np.diff(np.log(y)) * 100
print(f'\n=== Quarterly Growth Rate (%) ===')
print(f'Mean:     {np.mean(growth):.2f}%')
print(f'Std Dev:  {np.std(growth):.2f}%')
print(f'Min:      {np.min(growth):.2f}%')
print(f'Max:      {np.max(growth):.2f}%')

In [ ]:
# GRAPH 1: Time series plot with growth rate
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Level
axes[0].plot(df.index, y, color='steelblue', linewidth=1.5)
axes[0].set_title('Brazil Real GDP (Level)', fontsize=14)
axes[0].set_ylabel('GDP (Index)')
axes[0].grid(True, alpha=0.3)

# Growth rate
colors = ['green' if g >= 0 else 'red' for g in growth]
axes[1].bar(df.index[1:], growth, color=colors, alpha=0.7, width=60)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axhline(y=np.mean(growth), color='steelblue', linestyle='--',
                linewidth=1, label=f'Mean: {np.mean(growth):.2f}%')
axes[1].set_title('Quarterly Growth Rate (%)', fontsize=14)
axes[1].set_ylabel('Growth (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Unit Root Tests

Before modeling, we need to determine the integration order of the series.
We use three complementary tests:

| Test | H0 | H1 | Interpretation |
|------|----|----|----------------|
| ADF  | Unit root | Stationary | Reject H0 => stationary |
| KPSS | Stationary | Unit root | Reject H0 => non-stationary |
| ZA   | Unit root (no break) | Stationary with structural break | Reject H0 => stationary with break |

In [ ]:
# ADF Test
adf_result = adf_test(y, regression='c', autolag='AIC')
print('=== ADF Test (H0: Unit Root) ===')
print(f'Test Statistic: {adf_result.test_statistic:.4f}')
print(f'P-value:        {adf_result.pvalue:.4f}')
print(f'Decision:       {"Reject H0 (Stationary)" if adf_result.reject else "Fail to reject H0 (Non-stationary)"}')
print(f'Critical Values: {adf_result.critical_values}')

In [ ]:
# KPSS Test
kpss_result = kpss_test(y, regression='c')
print('=== KPSS Test (H0: Stationary) ===')
print(f'Test Statistic: {kpss_result.test_statistic:.4f}')
print(f'P-value:        {kpss_result.pvalue:.4f}')
print(f'Decision:       {"Reject H0 (Non-stationary)" if kpss_result.reject else "Fail to reject H0 (Stationary)"}')
print(f'Critical Values: {kpss_result.critical_values}')

In [ ]:
# Zivot-Andrews Test (allows for one structural break)
za_result = zivot_andrews_test(y, model='c', trim=0.15)
print('=== Zivot-Andrews Test (H0: Unit root without break) ===')
print(f'Test Statistic: {za_result.test_statistic:.4f}')
print(f'P-value:        {za_result.pvalue:.4f}')
print(f'Decision:       {"Reject H0" if za_result.reject else "Fail to reject H0"}')
print(f'Critical Values: {za_result.critical_values}')

In [ ]:
# Test on first differences
dy = np.diff(y)

print('=== Tests on First Differences (d=1) ===')
adf_diff = adf_test(dy, regression='c', autolag='AIC')
kpss_diff = kpss_test(dy, regression='c')

print(f'\nADF on diff(GDP):')
print(f'  Statistic: {adf_diff.test_statistic:.4f}, p-value: {adf_diff.pvalue:.4f}')
print(f'  Decision:  {"Stationary" if adf_diff.reject else "Non-stationary"}')

print(f'\nKPSS on diff(GDP):')
print(f'  Statistic: {kpss_diff.test_statistic:.4f}, p-value: {kpss_diff.pvalue:.4f}')
print(f'  Decision:  {"Non-stationary" if kpss_diff.reject else "Stationary"}')

print(f'\n=> Conclusion: GDP is I(1) - needs differencing for stationarity')

---
## 3. STL Decomposition

Seasonal-Trend decomposition using LOESS separates the series into three components:
- **Trend**: long-run movement
- **Seasonal**: recurring quarterly pattern
- **Remainder**: irregular/residual component

In [ ]:
# STL decomposition (period=4 for quarterly data)
stl = STL(period=4, seasonal=7, robust=True)
stl_result = stl.fit(y)

print('=== STL Decomposition ===')
print(f'Period:    {stl_result.period}')
print(f'Trend range:    [{stl_result.trend.min():.2f}, {stl_result.trend.max():.2f}]')
print(f'Seasonal range: [{stl_result.seasonal.min():.2f}, {stl_result.seasonal.max():.2f}]')
print(f'Remainder std:  {stl_result.remainder.std():.4f}')

In [ ]:
# GRAPH 2: STL decomposition plot
fig = plot_decomposition(
    results=stl_result,
    title='STL Decomposition - Brazil GDP',
    figsize=(12, 10)
)
plt.tight_layout()
plt.show()

---
## 4. Economic Filters

Filters extract the trend and cyclical components of the series.
We compare two widely used approaches:

- **Hodrick-Prescott (HP) filter**: minimizes a loss function that penalizes deviations from trend. Lambda=1600 for quarterly data.
- **Hamilton filter**: regression-based approach that avoids the end-of-sample bias problem of the HP filter.

In [ ]:
# HP Filter
hp_trend, hp_cycle = hp_filter(y, frequency='quarterly')

print('=== HP Filter (lambda=1600) ===')
print(f'Cycle std:  {hp_cycle.std():.4f}')
print(f'Cycle mean: {hp_cycle.mean():.4f}')
print(f'Max expansion: {hp_cycle.max():.4f}')
print(f'Max contraction: {hp_cycle.min():.4f}')

In [ ]:
# GRAPH 3: HP Filter - Trend and Cycle
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Trend
axes[0].plot(df.index, y, label='GDP', color='steelblue', alpha=0.7)
axes[0].plot(df.index, hp_trend, label='HP Trend', color='red', linewidth=2)
axes[0].set_title('HP Filter: Trend Extraction', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cycle
axes[1].fill_between(df.index, hp_cycle, 0,
                     where=hp_cycle >= 0, color='green', alpha=0.3, label='Expansion')
axes[1].fill_between(df.index, hp_cycle, 0,
                     where=hp_cycle < 0, color='red', alpha=0.3, label='Contraction')
axes[1].plot(df.index, hp_cycle, color='black', linewidth=0.8)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_title('HP Filter: Business Cycle', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Hamilton Filter
ham_trend, ham_cycle = hamilton_filter(y, h=4, p=4)

print('=== Hamilton Filter (h=4, p=4) ===')
# Hamilton filter returns shorter arrays (loses h+p-1 observations)
valid = ~np.isnan(ham_cycle)
print(f'Valid observations: {valid.sum()}')
print(f'Cycle std:  {np.nanstd(ham_cycle):.4f}')
print(f'Cycle mean: {np.nanmean(ham_cycle):.4f}')

In [ ]:
# GRAPH 4: Comparison of HP vs Hamilton cycles
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df.index, hp_cycle, label='HP Cycle', color='steelblue', linewidth=1.5)
ax.plot(df.index[:len(ham_cycle)], ham_cycle, label='Hamilton Cycle',
        color='darkorange', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('HP vs Hamilton Filter: Cycle Comparison', fontsize=14)
ax.set_ylabel('Cycle Component')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Model Fitting

We now fit four different models and compare their performance:

1. **ARIMA(1,1,1)**: Autoregressive Integrated Moving Average
2. **ARIMA(2,1,2)**: Higher-order ARIMA
3. **ETS(A,A,N)**: Exponential Smoothing with additive error and trend
4. **Auto-ARIMA**: Automatic model selection

We split the data into training (all but last 8 quarters) and test sets.

In [ ]:
# Train/test split
h = 8  # forecast horizon = 8 quarters (2 years)
y_train = y[:-h]
y_test = y[-h:]

print(f'Training set: {len(y_train)} observations')
print(f'Test set:     {len(y_test)} observations ({h} quarters)')
print(f'Train period: {df.index[0].date()} to {df.index[-h-1].date()}')
print(f'Test period:  {df.index[-h].date()} to {df.index[-1].date()}')

In [ ]:
# Model 1: ARIMA(1,1,1)
model_arima = ARIMA(order=(1, 1, 1))
result_arima = model_arima.fit(y_train)

print('=== ARIMA(1,1,1) ===')
print(result_arima.summary())

In [ ]:
# Model 2: ARIMA(2,1,2)
model_arima2 = ARIMA(order=(2, 1, 2))
result_arima2 = model_arima2.fit(y_train)

print('=== ARIMA(2,1,2) ===')
print(result_arima2.summary())

In [ ]:
# Model 3: ETS(A,A,N) - Additive error, additive trend, no seasonality
model_ets = ETS(error='A', trend='A', seasonal='N')
result_ets = model_ets.fit(y_train)

print('=== ETS(A,A,N) ===')
print(result_ets.summary())

In [ ]:
# Model 4: Auto-ARIMA (automatic order selection)
result_auto = auto_arima(
    y_train,
    seasonal=False,
    max_p=5, max_q=5, max_d=2,
    information_criterion='aicc',
    stepwise=True,
    trace=True
)

print('\n=== Auto-ARIMA Selected Model ===')
print(result_auto.summary())

---
## 6. Model Comparison

We compare models using:
- **In-sample**: AIC, BIC (information criteria)
- **Out-of-sample**: RMSE, MAE on the test set

In [ ]:
# Generate out-of-sample forecasts
models = {
    'ARIMA(1,1,1)': result_arima,
    'ARIMA(2,1,2)': result_arima2,
    'ETS(A,A,N)': result_ets,
    f'Auto-ARIMA ({result_auto.model_name})': result_auto,
}

comparison = []
forecasts = {}

for name, result in models.items():
    # Forecast h steps ahead
    fc = result.forecast(steps=h)
    fc_mean = fc['mean']
    forecasts[name] = fc_mean
    
    # Compute out-of-sample metrics
    errors = y_test - fc_mean
    rmse = np.sqrt(np.mean(errors**2))
    mae = np.mean(np.abs(errors))
    mape = np.mean(np.abs(errors / y_test)) * 100
    
    comparison.append({
        'Model': name,
        'AIC': result.aic,
        'BIC': result.bic,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE (%)': mape
    })

comp_df = pd.DataFrame(comparison).set_index('Model')
print('=== Model Comparison ===')
print(comp_df.round(4).to_string())
print(f'\nBest model (lowest RMSE): {comp_df["RMSE"].idxmin()}')

In [ ]:
# GRAPH 5: Out-of-sample forecast comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Plot training data (last 20 obs) and test data
n_show = 20
idx_train = df.index[-(h + n_show):-h]
idx_test = df.index[-h:]

ax.plot(idx_train, y_train[-n_show:], color='black', linewidth=1.5, label='Training')
ax.plot(idx_test, y_test, color='black', linewidth=2, linestyle='--', label='Actual')

colors = ['steelblue', 'darkorange', 'green', 'red']
for (name, fc_mean), color in zip(forecasts.items(), colors):
    ax.plot(idx_test, fc_mean, color=color, linewidth=1.5, label=name, marker='o', markersize=3)

ax.axvline(x=idx_test[0], color='gray', linestyle=':', alpha=0.5)
ax.set_title('Out-of-Sample Forecast Comparison', fontsize=14)
ax.set_ylabel('GDP')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# GRAPH 6: Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics = ['AIC', 'BIC', 'RMSE']
for ax, metric in zip(axes, metrics):
    values = comp_df[metric]
    colors_bar = ['green' if v == values.min() else 'steelblue' for v in values]
    bars = ax.barh(range(len(values)), values, color=colors_bar, alpha=0.8)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(comp_df.index, fontsize=8)
    ax.set_title(metric, fontsize=12)
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Model Comparison Metrics (green = best)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Diagnostics of the Best Model

In [ ]:
# Select the best model based on RMSE
best_name = comp_df['RMSE'].idxmin()
best_result = models[best_name]

print(f'Best model: {best_name}')
print(f'AIC:  {best_result.aic:.4f}')
print(f'BIC:  {best_result.bic:.4f}')

# Residual diagnostics
resid = best_result.residuals
lb = ljung_box_test(resid, lags=10)
jb = jarque_bera_test(resid)

print(f'\n=== Residual Diagnostics ===')
print(f'Ljung-Box (lag=10): stat={lb.test_statistic:.4f}, p={lb.pvalue:.4f}')
print(f'  -> {"No autocorrelation" if not lb.reject else "Autocorrelation detected"}')
print(f'Jarque-Bera:        stat={jb.test_statistic:.4f}, p={jb.pvalue:.4f}')
print(f'  -> {"Normal residuals" if not jb.reject else "Non-normal residuals"}')

In [ ]:
# GRAPH 7: Residual diagnostics plot
fig = plot_diagnostics(
    results=best_result,
    lags=16,
    title=f'Residual Diagnostics - {best_name}',
    figsize=(12, 8)
)
plt.tight_layout()
plt.show()

---
## 8. Forecasting with the Best Model

Now we refit the best model on the full dataset and generate forward-looking forecasts.

In [ ]:
# Refit best model on full data
best_model_full = ARIMA(order=best_result.model.order)
best_result_full = best_model_full.fit(y)

# Forecast 12 quarters ahead (3 years)
h_fwd = 12
fc_full = best_result_full.forecast(steps=h_fwd, alpha=0.05)

print(f'=== {h_fwd}-Quarter Ahead Forecast ===')
fc_df = pd.DataFrame({
    'Forecast': fc_full['mean'],
    'Lower 95%': fc_full['lower'],
    'Upper 95%': fc_full['upper']
}, index=pd.date_range(start=df.index[-1] + pd.DateOffset(months=3),
                       periods=h_fwd, freq='QS'))
print(fc_df.round(2).to_string())

In [ ]:
# GRAPH 8: Final forecast with confidence intervals
fig = plot_forecast(
    results=best_result_full,
    steps=h_fwd,
    alpha=0.05,
    history_periods=40,
    title=f'GDP Forecast - {best_name}',
    figsize=(12, 6)
)
plt.tight_layout()
plt.show()

In [ ]:
# GRAPH 9: Cumulative forecast performance
fig, ax = plt.subplots(figsize=(12, 5))

for name, fc_mean in forecasts.items():
    cum_error = np.cumsum(np.abs(y_test - fc_mean))
    ax.plot(range(1, h + 1), cum_error, marker='o', label=name, linewidth=1.5)

ax.set_xlabel('Forecast Horizon (quarters)')
ax.set_ylabel('Cumulative Absolute Error')
ax.set_title('Cumulative Forecast Error by Model', fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Integration Between Modules

This section demonstrates how to chain chronobox functions from different modules
into a cohesive analysis pipeline.

In [ ]:
# Example: Complete pipeline in a few lines
# Step 1: Test stationarity
from chronobox.tests_stat import adf_test, kpss_test

is_stationary = adf_test(y).reject and not kpss_test(y).reject
d = 0 if is_stationary else 1
print(f'Integration order: d = {d}')

# Step 2: Extract cycle with HP filter
from chronobox.filters import hp_filter
trend, cycle = hp_filter(y, frequency='quarterly')

# Step 3: Auto-select and fit model
from chronobox import auto_arima
result = auto_arima(y, seasonal=False, d=d, trace=False)
print(f'Selected model: {result.model_name}')
print(f'AIC: {result.aic:.2f}')

# Step 4: Forecast
fc = result.forecast(steps=4)
print(f'Next 4 quarters forecast: {np.round(fc["mean"], 2)}')

# Step 5: Diagnostics
from chronobox.tests_stat import ljung_box_test
lb = ljung_box_test(result.residuals, lags=8)
print(f'Residual autocorrelation (Ljung-Box p={lb.pvalue:.3f}): '
      f'{"OK" if not lb.reject else "Problem"}')

---
## Exercicio

**Replicate this pipeline using the `airline.csv` dataset** (monthly airline passengers, 1949-1960).

The airline data has important differences from GDP:
- Monthly frequency (not quarterly)
- Strong seasonal pattern (summer peaks)
- Multiplicative seasonality (variance increases with level)

Tasks:
1. Load `airline.csv` and plot the series
2. Run ADF and KPSS tests on the raw and log-transformed series
3. Apply STL decomposition with `period=12`
4. Fit at least 4 models: ARIMA, seasonal ARIMA, ETS with seasonality, Auto-ARIMA with `seasonal=True, m=12`
5. Compare models using an 12-month holdout set
6. Which model performs best? Is the seasonal component important?

In [ ]:
# === YOUR CODE HERE ===

# 1. Load airline data
# airline = pd.read_csv(os.path.join(data_dir, 'airline.csv'), parse_dates=['date'])
# airline = airline.set_index('date')
# y_air = airline['passengers'].values

# 2. Unit root tests
# adf_air = adf_test(y_air)
# adf_air_log = adf_test(np.log(y_air))

# 3. STL decomposition
# stl_air = STL(period=12, robust=True)
# stl_air_result = stl_air.fit(np.log(y_air))
# plot_decomposition(results=stl_air_result)

# 4. Model fitting (use seasonal=True, m=12)
# model_sarima = ARIMA(order=(1,1,1), seasonal_order=(1,1,1,12))
# result_sarima = model_sarima.fit(y_air_train)

# model_ets_s = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
# result_ets_s = model_ets_s.fit(y_air_train)

# result_auto_s = auto_arima(y_air_train, seasonal=True, m=12)

# 5. Compare out-of-sample RMSE

# 6. Conclusions
print('Complete the exercise above!')